# Open-Meteo Weather Data Validation

This notebook validates Open-Meteo as the weather-data source for the selected Zafar Memon DHA PM2.5 sensor.

The notebook will:

1. Retrieve hourly historical weather data for the selected location
2. Inspect the returned schema and measurement units
3. Validate hourly completeness and timestamp continuity
4. Check missing, invalid, and duplicate observations
5. Retrieve forecast data using the same weather variables
6. Confirm that historical and forecast responses can be transformed into one consistent schema
7. Save validated weather datasets for later joining with OpenAQ PM2.5 data

## 1. Location, Date Range, and API Configuration

The weather coordinates correspond to the selected Zafar Memon DHA monitoring location.

UTC is used throughout the project so that Open-Meteo weather timestamps can later be joined directly with the hourly OpenAQ PM2.5 observations.

In [ ]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

In [3]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "data" / "reports"

In [4]:
LATITUDE = 24.814741
LONGITUDE = 67.067062

LOCATION_NAME = "Zafar Memon DHA, Karachi"
TIMEZONE = "UTC"

OUTPUT_DIR = (
    REPORTS_DIR / "open_meteo_validation"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Match the common evaluation period used for the selected OpenAQ sensors.
HISTORICAL_START = "2025-07-08"
HISTORICAL_END = "2026-07-23"

HISTORICAL_URL = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

WEATHER_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "surface_pressure",
    "precipitation",
    "rain",
    "cloud_cover",
    "visibility",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
]

print(f"Location: {LOCATION_NAME}")
print(f"Coordinates: {LATITUDE}, {LONGITUDE}")
print(f"Timezone: {TIMEZONE}")
print(f"Historical period: {HISTORICAL_START} to {HISTORICAL_END}")
print(f"Output directory: {OUTPUT_DIR}")

Location: Zafar Memon DHA, Karachi
Coordinates: 24.814741, 67.067062
Timezone: UTC
Historical period: 2025-07-08 to 2026-07-23
Output directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/open_meteo_validation


## 2. Open-Meteo API Client

This section creates a reusable HTTP session for Open-Meteo requests.

The request helper provides:

- connection reuse
- request timeouts
- automatic retries
- exponential backoff
- rate-limit handling
- temporary server-error handling
- response structure validation

In [5]:
class WeatherAPIError(RuntimeError):
    """Raised when an Open-Meteo request fails."""


session = requests.Session()
session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": "karachi-pm25-weather-feasibility/1.0",
    }
)


def request_json(
    url: str,
    params: dict[str, Any],
    max_retries: int = 5,
    timeout_seconds: int = 60,
) -> dict[str, Any]:
    """
    Send a GET request to Open-Meteo with retry handling.

    Retries are applied to rate limits, request timeouts,
    and temporary server-side failures.
    """
    last_error: Exception | None = None

    retryable_status_codes = {
        408,
        429,
        500,
        502,
        503,
        504,
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=timeout_seconds,
            )

            if response.status_code in retryable_status_codes:
                if response.status_code == 429:
                    wait_seconds = int(
                        response.headers.get(
                            "Retry-After",
                            min(2**attempt, 30),
                        )
                    )
                else:
                    wait_seconds = min(2**attempt, 30)

                last_error = WeatherAPIError(
                    f"Temporary HTTP {response.status_code} "
                    f"from {response.url}"
                )

                if attempt < max_retries:
                    print(
                        f"Temporary error {response.status_code}. "
                        f"Retrying in {wait_seconds}s..."
                    )
                    time.sleep(wait_seconds)
                    continue

                break

            response.raise_for_status()

            payload = response.json()

            if not isinstance(payload, dict):
                raise WeatherAPIError(
                    f"Unexpected response structure from "
                    f"{response.url}"
                )

            if payload.get("error"):
                reason = payload.get(
                    "reason",
                    "Open-Meteo returned an unspecified error.",
                )

                raise WeatherAPIError(str(reason))

            return payload

        except (
            requests.RequestException,
            ValueError,
            WeatherAPIError,
        ) as exc:
            last_error = exc

            if attempt < max_retries:
                wait_seconds = min(2**attempt, 30)

                print(
                    f"Request failed: {exc}. "
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)

    raise WeatherAPIError(
        f"Request failed after {max_retries} attempts. "
        f"URL: {url}. Last error: {last_error}"
    )

## 3. Convert Hourly API Responses into a DataFrame

Open-Meteo returns hourly timestamps and weather variables as parallel arrays.

This function converts the response into a structured DataFrame, validates that every weather variable has the same number of values as the timestamp array, converts numeric fields safely, and stores source metadata and measurement units.

In [6]:
def hourly_payload_to_dataframe(
    payload: dict[str, Any],
) -> pd.DataFrame:
    """
    Convert an Open-Meteo hourly response into a normalized DataFrame.

    The function validates array lengths, converts timestamps to UTC,
    converts weather variables to numeric values, and preserves source
    metadata and units.
    """
    hourly = payload.get("hourly")
    hourly_units = payload.get("hourly_units") or {}

    if not isinstance(hourly, dict):
        raise WeatherAPIError(
            "Response does not contain a valid hourly data object."
        )

    times = hourly.get("time")

    if not isinstance(times, list):
        raise WeatherAPIError(
            "Hourly response does not contain a valid time array."
        )

    if not times:
        return pd.DataFrame()

    dataframe = pd.DataFrame(
        {
            "datetime_utc": pd.to_datetime(
                times,
                utc=True,
                errors="coerce",
            )
        }
    )

    for variable, values in hourly.items():
        if variable == "time":
            continue

        if not isinstance(values, list):
            raise WeatherAPIError(
                f"Hourly variable {variable!r} is not a list."
            )

        if len(values) != len(dataframe):
            raise WeatherAPIError(
                f"Length mismatch for variable {variable!r}: "
                f"{len(values)} values for "
                f"{len(dataframe)} timestamps."
            )

        dataframe[variable] = pd.to_numeric(
            values,
            errors="coerce",
        )

    invalid_timestamp_count = int(
        dataframe["datetime_utc"].isna().sum()
    )

    if invalid_timestamp_count:
        print(
            f"Dropping {invalid_timestamp_count} row(s) "
            "with invalid timestamps."
        )

    dataframe = (
        dataframe.dropna(subset=["datetime_utc"])
        .sort_values("datetime_utc")
        .reset_index(drop=True)
    )

    dataframe["source_latitude"] = payload.get("latitude")
    dataframe["source_longitude"] = payload.get("longitude")
    dataframe["source_elevation"] = payload.get("elevation")
    dataframe["source_timezone"] = payload.get("timezone")
    dataframe["utc_offset_seconds"] = payload.get(
        "utc_offset_seconds"
    )

    dataframe.attrs["hourly_units"] = hourly_units

    return dataframe

## 4. Test the Historical Weather Endpoint

Before downloading the complete historical period, a smaller date range is requested to verify:

- API connectivity
- requested variable availability
- hourly timestamp formatting
- response units
- first and last returned timestamps
- successful conversion into a DataFrame

In [7]:
# Use a small period first to validate the API response and schema.
HISTORICAL_TEST_START = "2026-06-24"
HISTORICAL_TEST_END = "2026-07-23"

historical_test_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": HISTORICAL_TEST_START,
    "end_date": HISTORICAL_TEST_END,
    "hourly": ",".join(WEATHER_VARIABLES),
    "timezone": TIMEZONE,
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
    "timeformat": "iso8601",
}

historical_test_payload = request_json(
    url=HISTORICAL_URL,
    params=historical_test_params,
)

historical_test_df = hourly_payload_to_dataframe(
    historical_test_payload
)

if historical_test_df.empty:
    raise WeatherAPIError(
        "The historical test request returned no hourly records."
    )

print(f"Rows returned: {len(historical_test_df)}")
print(
    "First timestamp: "
    f"{historical_test_df['datetime_utc'].min()}"
)
print(
    "Last timestamp: "
    f"{historical_test_df['datetime_utc'].max()}"
)

expected_columns = {
    "datetime_utc",
    *WEATHER_VARIABLES,
}

missing_columns = (
    expected_columns - set(historical_test_df.columns)
)

if missing_columns:
    print(
        "\nRequested variables missing from the response:"
    )
    print(sorted(missing_columns))
else:
    print("\nAll requested weather variables were returned.")

print("\nHourly units:")
print(
    json.dumps(
        historical_test_payload.get(
            "hourly_units",
            {},
        ),
        indent=2,
    )
)

display(historical_test_df.head())
display(historical_test_df.tail())

Rows returned: 720
First timestamp: 2026-06-24 00:00:00+00:00
Last timestamp: 2026-07-23 23:00:00+00:00

All requested weather variables were returned.

Hourly units:
{
  "time": "iso8601",
  "temperature_2m": "\u00b0C",
  "relative_humidity_2m": "%",
  "dew_point_2m": "\u00b0C",
  "surface_pressure": "hPa",
  "precipitation": "mm",
  "rain": "mm",
  "cloud_cover": "%",
  "visibility": "undefined",
  "wind_speed_10m": "km/h",
  "wind_direction_10m": "\u00b0",
  "wind_gusts_10m": "km/h"
}


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds
0,2026-06-24 00:00:00+00:00,28.9,79,24.9,1000.5,0.0,0.0,95,NaN,16.8,266,41.4,24.850615,67.08915,5.0,GMT,0
1,2026-06-24 01:00:00+00:00,29.0,76,24.4,1000.8,0.0,0.0,99,NaN,17.3,273,38.5,24.850615,67.08915,5.0,GMT,0
2,2026-06-24 02:00:00+00:00,29.0,77,24.5,1001.2,0.1,0.1,100,NaN,17.8,271,42.1,24.850615,67.08915,5.0,GMT,0
3,2026-06-24 03:00:00+00:00,29.5,74,24.4,1001.8,0.0,0.0,98,NaN,17.3,271,42.1,24.850615,67.08915,5.0,GMT,0
4,2026-06-24 04:00:00+00:00,30.2,70,24.2,1002.1,0.0,0.0,100,NaN,18.9,261,45.0,24.850615,67.08915,5.0,GMT,0


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds
715,2026-07-23 19:00:00+00:00,29.2,82,25.8,998.8,0.0,0.0,41,NaN,10.4,238,27.7,24.850615,67.08915,5.0,GMT,0
716,2026-07-23 20:00:00+00:00,28.9,83,25.7,998.6,0.0,0.0,43,NaN,8.8,242,25.2,24.850615,67.08915,5.0,GMT,0
717,2026-07-23 21:00:00+00:00,28.7,83,25.6,997.9,0.0,0.0,88,NaN,8.1,245,21.2,24.850615,67.08915,5.0,GMT,0
718,2026-07-23 22:00:00+00:00,28.7,82,25.4,997.8,0.0,0.0,98,NaN,7.0,249,20.2,24.850615,67.08915,5.0,GMT,0
719,2026-07-23 23:00:00+00:00,28.6,83,25.4,997.7,0.0,0.0,45,NaN,5.2,254,16.9,24.850615,67.08915,5.0,GMT,0


## 5. Validate Historical Weather Completeness

The historical test dataset is evaluated against the complete expected hourly timeline.

The validation checks:

- expected and received hourly records
- duplicate timestamps
- missing weather values
- incomplete hourly rows
- longest continuous incomplete period
- first and last available timestamps
- missing values for each requested weather variable

An hour is considered complete only when all required weather variables contain valid values.

In [8]:
def longest_missing_run(mask: pd.Series) -> int:
    """Return the longest consecutive sequence of True values."""
    longest = 0
    current = 0

    for value in mask:
        if bool(value):
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


def evaluate_weather_dataframe(
    dataframe: pd.DataFrame,
    expected_start: pd.Timestamp,
    expected_end_exclusive: pd.Timestamp,
    dataset_name: str,
) -> dict[str, Any]:
    """
    Evaluate hourly weather completeness over a fixed UTC period.

    An hour is considered complete only when every requested weather
    variable is present and non-null.
    """
    if expected_start >= expected_end_exclusive:
        raise ValueError(
            "expected_start must be earlier than expected_end_exclusive."
        )

    expected_index = pd.date_range(
        start=expected_start.floor("h"),
        end=expected_end_exclusive.floor("h"),
        freq="h",
        inclusive="left",
    )

    expected_hours = len(expected_index)

    base_report = {
        "dataset": dataset_name,
        "period_start": expected_start.isoformat(),
        "period_end_exclusive": expected_end_exclusive.isoformat(),
        "expected_hours": expected_hours,
    }

    if dataframe.empty:
        return {
            **base_report,
            "received_rows": 0,
            "complete_hours": 0,
            "missing_or_incomplete_hours": expected_hours,
            "completeness_percent": 0.0,
            "duplicate_hours": 0,
            "invalid_timestamp_rows": 0,
            "longest_incomplete_gap_hours": expected_hours,
            "first_timestamp": None,
            "last_timestamp": None,
            "missing_variables": WEATHER_VARIABLES.copy(),
            "missing_values_by_variable": {
                variable: expected_hours
                for variable in WEATHER_VARIABLES
            },
        }

    if "datetime_utc" not in dataframe.columns:
        raise ValueError(
            "Weather DataFrame is missing the datetime_utc column."
        )

    raw = dataframe.copy()

    invalid_timestamp_rows = int(
        raw["datetime_utc"].isna().sum()
    )

    clean = raw.dropna(
        subset=["datetime_utc"]
    ).copy()

    clean["hour_utc"] = (
        clean["datetime_utc"].dt.floor("h")
    )

    duplicate_hours = int(
        clean.duplicated(
            subset=["hour_utc"],
            keep=False,
        ).sum()
    )

    hourly = (
        clean.sort_values("datetime_utc")
        .drop_duplicates(
            subset=["hour_utc"],
            keep="last",
        )
        .set_index("hour_utc")
        .reindex(expected_index)
    )

    available_weather_columns = [
        variable
        for variable in WEATHER_VARIABLES
        if variable in hourly.columns
    ]

    missing_variables = [
        variable
        for variable in WEATHER_VARIABLES
        if variable not in hourly.columns
    ]

    if missing_variables:
        # Missing requested variables make every hour incomplete.
        row_has_all_required_values = pd.Series(
            False,
            index=expected_index,
            dtype=bool,
        )
    else:
        row_has_all_required_values = (
            hourly[available_weather_columns]
            .notna()
            .all(axis=1)
        )

    incomplete_rows = ~row_has_all_required_values

    per_column_missing = {
        variable: (
            int(hourly[variable].isna().sum())
            if variable in hourly.columns
            else expected_hours
        )
        for variable in WEATHER_VARIABLES
    }

    complete_hours = int(
        row_has_all_required_values.sum()
    )

    completeness_percent = (
        round(
            complete_hours / expected_hours * 100,
            2,
        )
        if expected_hours
        else 0.0
    )

    first_timestamp = (
        clean["datetime_utc"].min()
        if not clean.empty
        else None
    )

    last_timestamp = (
        clean["datetime_utc"].max()
        if not clean.empty
        else None
    )

    return {
        **base_report,
        "received_rows": len(raw),
        "complete_hours": complete_hours,
        "missing_or_incomplete_hours": (
            expected_hours - complete_hours
        ),
        "completeness_percent": completeness_percent,
        "duplicate_hours": duplicate_hours,
        "invalid_timestamp_rows": invalid_timestamp_rows,
        "longest_incomplete_gap_hours": (
            longest_missing_run(incomplete_rows)
        ),
        "first_timestamp": (
            first_timestamp.isoformat()
            if first_timestamp is not None
            else None
        ),
        "last_timestamp": (
            last_timestamp.isoformat()
            if last_timestamp is not None
            else None
        ),
        "missing_variables": missing_variables,
        "missing_values_by_variable": per_column_missing,
    }

In [9]:
historical_test_start = pd.Timestamp(
    f"{HISTORICAL_TEST_START}T00:00:00Z"
)

# Open-Meteo end_date is inclusive, so evaluation ends at the
# beginning of the following day.
historical_test_end_exclusive = (
    pd.Timestamp(f"{HISTORICAL_TEST_END}T00:00:00Z")
    + pd.Timedelta(days=1)
)

historical_test_report = evaluate_weather_dataframe(
    dataframe=historical_test_df,
    expected_start=historical_test_start,
    expected_end_exclusive=historical_test_end_exclusive,
    dataset_name="Open-Meteo historical weather test",
)

print(
    json.dumps(
        historical_test_report,
        indent=2,
    )
)

{
  "dataset": "Open-Meteo historical weather test",
  "period_start": "2026-06-24T00:00:00+00:00",
  "period_end_exclusive": "2026-07-24T00:00:00+00:00",
  "expected_hours": 720,
  "received_rows": 720,
  "complete_hours": 0,
  "missing_or_incomplete_hours": 720,
  "completeness_percent": 0.0,
  "duplicate_hours": 0,
  "invalid_timestamp_rows": 0,
  "longest_incomplete_gap_hours": 720,
  "first_timestamp": "2026-06-24T00:00:00+00:00",
  "last_timestamp": "2026-07-23T23:00:00+00:00",
  "missing_variables": [],
  "missing_values_by_variable": {
    "temperature_2m": 0,
    "relative_humidity_2m": 0,
    "dew_point_2m": 0,
    "surface_pressure": 0,
    "precipitation": 0,
    "rain": 0,
    "cloud_cover": 0,
    "visibility": 720,
    "wind_speed_10m": 0,
    "wind_direction_10m": 0,
    "wind_gusts_10m": 0
  }
}


## 6. Split the Historical Period into Monthly Requests

The complete historical period is divided into calendar-month chunks.

Using smaller requests makes the download process easier to monitor and reduces the impact of temporary API failures. Each generated interval includes both its start and end date.

In [10]:
from collections.abc import Iterator


def generate_month_chunks(
    start_date: str,
    end_date: str,
) -> Iterator[tuple[str, str]]:
    """
    Generate inclusive calendar-month date ranges.

    Example:
        2025-07-08 to 2025-07-31
        2025-08-01 to 2025-08-31
    """
    current = pd.Timestamp(start_date)
    final = pd.Timestamp(end_date)

    if pd.isna(current) or pd.isna(final):
        raise ValueError(
            "start_date and end_date must be valid dates."
        )

    if current > final:
        raise ValueError(
            "start_date must be earlier than or equal to end_date."
        )

    while current <= final:
        month_end = min(
            current + pd.offsets.MonthEnd(0),
            final,
        )

        yield (
            current.strftime("%Y-%m-%d"),
            month_end.strftime("%Y-%m-%d"),
        )

        current = month_end + pd.Timedelta(days=1)

## 7. Download One Historical Weather Chunk

This function requests hourly Open-Meteo archive data for one bounded date range.

Each returned row is labelled with:

- the requested chunk start date
- the requested chunk end date
- the weather-data source

These fields make it easier to trace records back to their original API request.

In [11]:
def fetch_historical_weather_chunk(
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """
    Download and normalize one historical Open-Meteo date chunk.
    """
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(WEATHER_VARIABLES),
        "timezone": TIMEZONE,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm",
        "timeformat": "iso8601",
    }

    payload = request_json(
        url=HISTORICAL_URL,
        params=params,
    )

    dataframe = hourly_payload_to_dataframe(payload)

    if dataframe.empty:
        print(
            f"No hourly weather data returned for "
            f"{start_date} to {end_date}."
        )

        return dataframe

    dataframe["requested_start_date"] = start_date
    dataframe["requested_end_date"] = end_date
    dataframe["weather_source"] = "open_meteo_archive"

    return dataframe

## 8. Download the Complete Historical Weather Period

The full historical period is downloaded month by month.

Each successful response is retained and combined into one DataFrame. Temporary failures for an individual month do not discard data already downloaded for other months.

The combined dataset remains unmodified during validation so that duplicate timestamps and other API-quality issues can be measured accurately.

In [12]:
historical_chunks: list[pd.DataFrame] = []
failed_historical_chunks: list[dict[str, str]] = []

chunks = list(
    generate_month_chunks(
        start_date=HISTORICAL_START,
        end_date=HISTORICAL_END,
    )
)

for index, (start_date, end_date) in enumerate(
    chunks,
    start=1,
):
    print(
        f"[{index}/{len(chunks)}] "
        f"Downloading {start_date} to {end_date}"
    )

    try:
        chunk_dataframe = fetch_historical_weather_chunk(
            start_date=start_date,
            end_date=end_date,
        )

        print(f"  Returned {len(chunk_dataframe)} rows")

        if not chunk_dataframe.empty:
            historical_chunks.append(chunk_dataframe)

    except WeatherAPIError as exc:
        print(
            f"  Failed to download {start_date} to {end_date}: {exc}"
        )

        failed_historical_chunks.append(
            {
                "start_date": start_date,
                "end_date": end_date,
                "error": str(exc),
            }
        )

    # Avoid sending requests too aggressively.
    time.sleep(0.2)

if failed_historical_chunks:
    failed_chunks_file = (
        OUTPUT_DIR / "historical_weather_failed_chunks.json"
    )

    with failed_chunks_file.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            failed_historical_chunks,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print(f"\nFailed chunks saved to: {failed_chunks_file}")

if not historical_chunks:
    raise WeatherAPIError(
        "No historical weather chunks were downloaded successfully."
    )

historical_weather_df = (
    pd.concat(
        historical_chunks,
        ignore_index=True,
    )
    .sort_values("datetime_utc")
    .reset_index(drop=True)
)

historical_output_file = (
    OUTPUT_DIR
    / "zafar_memon_dha_historical_weather_raw.csv"
)

historical_weather_df.to_csv(
    historical_output_file,
    index=False,
)

print(f"\nTotal downloaded rows: {len(historical_weather_df)}")
print(f"Saved raw historical data to: {historical_output_file}")

display(historical_weather_df.head())
display(historical_weather_df.tail())

[1/13] Downloading 2025-07-08 to 2025-07-31
  Returned 576 rows
[2/13] Downloading 2025-08-01 to 2025-08-31
  Returned 744 rows
[3/13] Downloading 2025-09-01 to 2025-09-30
  Returned 720 rows
[4/13] Downloading 2025-10-01 to 2025-10-31
  Returned 744 rows
[5/13] Downloading 2025-11-01 to 2025-11-30
  Returned 720 rows
[6/13] Downloading 2025-12-01 to 2025-12-31
  Returned 744 rows
[7/13] Downloading 2026-01-01 to 2026-01-31
  Returned 744 rows
[8/13] Downloading 2026-02-01 to 2026-02-28
  Returned 672 rows
[9/13] Downloading 2026-03-01 to 2026-03-31
  Returned 744 rows
[10/13] Downloading 2026-04-01 to 2026-04-30
  Returned 720 rows
[11/13] Downloading 2026-05-01 to 2026-05-31
  Returned 744 rows
[12/13] Downloading 2026-06-01 to 2026-06-30
  Returned 720 rows
[13/13] Downloading 2026-07-01 to 2026-07-23
  Returned 552 rows

Total downloaded rows: 9144
Saved raw historical data to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/open_meteo_validation/zafar_memon_dha_histor

,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds,requested_start_date,requested_end_date,weather_source
0,2025-07-08 00:00:00+00:00,28.5,87,26.2,997.3,0.1,0.1,49,NaN,3.6,225,8.6,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
1,2025-07-08 01:00:00+00:00,29.1,82,25.8,997.9,0.0,0.0,98,NaN,1.5,263,5.4,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
2,2025-07-08 02:00:00+00:00,29.4,80,25.6,998.5,0.0,0.0,82,NaN,0.7,270,5.0,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
3,2025-07-08 03:00:00+00:00,30.2,75,25.3,998.9,0.0,0.0,79,NaN,2.2,228,10.1,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
4,2025-07-08 04:00:00+00:00,31.1,70,24.9,999.3,0.0,0.0,32,NaN,3.2,199,13.7,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds,requested_start_date,requested_end_date,weather_source
9139,2026-07-23 19:00:00+00:00,29.2,82,25.8,998.8,0.0,0.0,41,NaN,10.4,238,27.7,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9140,2026-07-23 20:00:00+00:00,28.9,83,25.7,998.6,0.0,0.0,43,NaN,8.8,242,25.2,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9141,2026-07-23 21:00:00+00:00,28.7,83,25.6,997.9,0.0,0.0,88,NaN,8.1,245,21.2,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9142,2026-07-23 22:00:00+00:00,28.7,82,25.4,997.8,0.0,0.0,98,NaN,7.0,249,20.2,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9143,2026-07-23 23:00:00+00:00,28.6,83,25.4,997.7,0.0,0.0,45,NaN,5.2,254,16.9,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive


## 9. Validate the Complete Historical Weather Dataset

The combined historical dataset is evaluated against the full expected hourly timeline.

The report measures:

- expected hourly records
- received rows
- complete hourly observations
- missing or incomplete hours
- duplicate timestamps
- longest incomplete period
- missing values for each weather variable
- first and last returned timestamps

The resulting validation report is saved as JSON for later review.

In [13]:
historical_start_timestamp = pd.Timestamp(
    f"{HISTORICAL_START}T00:00:00Z"
)

# Open-Meteo end_date is inclusive, so the expected interval
# ends at midnight on the following day.
historical_end_exclusive = (
    pd.Timestamp(f"{HISTORICAL_END}T00:00:00Z")
    + pd.Timedelta(days=1)
)

historical_full_report = evaluate_weather_dataframe(
    dataframe=historical_weather_df,
    expected_start=historical_start_timestamp,
    expected_end_exclusive=historical_end_exclusive,
    dataset_name="Open-Meteo full historical weather",
)

print(
    json.dumps(
        historical_full_report,
        indent=2,
    )
)

historical_report_file = (
    OUTPUT_DIR / "historical_weather_report.json"
)

with historical_report_file.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        historical_full_report,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"\nHistorical validation report saved to: "
    f"{historical_report_file}"
)

{
  "dataset": "Open-Meteo full historical weather",
  "period_start": "2025-07-08T00:00:00+00:00",
  "period_end_exclusive": "2026-07-24T00:00:00+00:00",
  "expected_hours": 9144,
  "received_rows": 9144,
  "complete_hours": 0,
  "missing_or_incomplete_hours": 9144,
  "completeness_percent": 0.0,
  "duplicate_hours": 0,
  "invalid_timestamp_rows": 0,
  "longest_incomplete_gap_hours": 9144,
  "first_timestamp": "2025-07-08T00:00:00+00:00",
  "last_timestamp": "2026-07-23T23:00:00+00:00",
  "missing_variables": [],
  "missing_values_by_variable": {
    "temperature_2m": 0,
    "relative_humidity_2m": 0,
    "dew_point_2m": 0,
    "surface_pressure": 0,
    "precipitation": 0,
    "rain": 0,
    "cloud_cover": 0,
    "visibility": 9144,
    "wind_speed_10m": 0,
    "wind_direction_10m": 0,
    "wind_gusts_10m": 0
  }
}

Historical validation report saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/open_meteo_validation/historical_weather_report.json


## 10. Retrieve the 72-Hour Weather Forecast

The project requires future weather conditions as inputs for the PM2.5 forecasting model.

This request retrieves the next 72 hourly weather observations using the same variables, units, coordinates, and UTC timezone as the historical weather dataset.

Using a consistent schema allows historical and forecast weather data to pass through the same preprocessing and feature-generation pipeline.

In [14]:
forecast_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": ",".join(WEATHER_VARIABLES),
    "forecast_hours": 72,
    "timezone": TIMEZONE,
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
    "timeformat": "iso8601",
}

forecast_payload = request_json(
    url=FORECAST_URL,
    params=forecast_params,
)

forecast_df = hourly_payload_to_dataframe(
    forecast_payload
)

if forecast_df.empty:
    raise WeatherAPIError(
        "The 72-hour forecast request returned no hourly records."
    )

forecast_df["weather_source"] = "open_meteo_forecast"

forecast_retrieved_at_utc = pd.Timestamp.now(
    tz="UTC"
).floor("s")

forecast_df["forecast_retrieved_at_utc"] = (
    forecast_retrieved_at_utc
)

forecast_output_file = (
    OUTPUT_DIR
    / "zafar_memon_dha_72_hour_weather_forecast.csv"
)

forecast_df.to_csv(
    forecast_output_file,
    index=False,
)

print(f"Forecast rows: {len(forecast_df)}")
print(
    "First forecast time: "
    f"{forecast_df['datetime_utc'].min()}"
)
print(
    "Last forecast time: "
    f"{forecast_df['datetime_utc'].max()}"
)
print(
    "Forecast retrieved at: "
    f"{forecast_retrieved_at_utc}"
)
print(f"Saved: {forecast_output_file}")

display(forecast_df.head())
display(forecast_df.tail())

Forecast rows: 72
First forecast time: 2026-07-25 11:00:00+00:00
Last forecast time: 2026-07-28 10:00:00+00:00
Forecast retrieved at: 2026-07-25 11:05:56+00:00
Saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/open_meteo_validation/zafar_memon_dha_72_hour_weather_forecast.csv


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds,weather_source,forecast_retrieved_at_utc
0,2026-07-25 11:00:00+00:00,30.5,73,25.1,999.4,0.0,0.0,100,18000.0,14.3,225,39.6,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
1,2026-07-25 12:00:00+00:00,29.9,75,25.1,999.0,0.0,0.0,100,18060.0,15.2,234,38.5,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
2,2026-07-25 13:00:00+00:00,29.8,75,25.0,999.5,0.0,0.0,100,18060.0,13.6,240,38.2,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
3,2026-07-25 14:00:00+00:00,29.5,77,25.0,999.9,0.0,0.0,100,18080.0,13.0,235,34.2,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
4,2026-07-25 15:00:00+00:00,29.1,79,25.1,1000.3,0.0,0.0,100,16180.0,11.9,241,32.4,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds,weather_source,forecast_retrieved_at_utc
67,2026-07-28 06:00:00+00:00,31.7,63,23.8,997.7,0.0,0.0,76,19740.0,15.2,248,39.2,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
68,2026-07-28 07:00:00+00:00,31.8,65,24.4,997.4,0.0,0.0,71,19740.0,14.8,241,39.2,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
69,2026-07-28 08:00:00+00:00,32.4,64,24.7,996.8,0.0,0.0,66,19740.0,14.5,231,39.6,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
70,2026-07-28 09:00:00+00:00,32.0,66,24.9,996.1,0.0,0.0,73,19740.0,16.0,225,42.8,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00
71,2026-07-28 10:00:00+00:00,31.3,69,24.9,995.5,0.0,0.0,83,17620.0,16.2,225,43.2,24.850615,67.08915,5.0,GMT,0,open_meteo_forecast,2026-07-25 11:05:56+00:00


## 11. Validate the 72-Hour Weather Forecast

The forecast dataset is checked to confirm that it is suitable for the PM2.5 prediction pipeline.

The validation checks:

- total returned forecast rows
- unique hourly timestamps
- duplicate timestamps
- missing requested variables
- missing values within each weather variable
- timestamp continuity
- full coverage of at least 72 consecutive hours
- completeness of all required weather features

In [15]:
EXPECTED_FORECAST_HOURS = 72


def evaluate_forecast(
    dataframe: pd.DataFrame,
) -> dict[str, Any]:
    """
    Validate the structure, completeness, and hourly continuity
    of an Open-Meteo forecast DataFrame.
    """
    base_report = {
        "required_forecast_hours": EXPECTED_FORECAST_HOURS,
    }

    if dataframe.empty:
        return {
            **base_report,
            "returned_rows": 0,
            "unique_forecast_hours": 0,
            "complete_hours": 0,
            "duplicate_hours": 0,
            "invalid_timestamp_rows": 0,
            "missing_variables": WEATHER_VARIABLES.copy(),
            "missing_values_by_variable": {
                variable: 0
                for variable in WEATHER_VARIABLES
            },
            "first_forecast_timestamp": None,
            "last_forecast_timestamp": None,
            "longest_missing_gap_hours": (
                EXPECTED_FORECAST_HOURS
            ),
            "covers_72_hours": False,
            "timestamps_are_hourly": False,
            "all_required_values_complete": False,
            "forecast_is_valid": False,
        }

    if "datetime_utc" not in dataframe.columns:
        raise ValueError(
            "Forecast DataFrame is missing the datetime_utc column."
        )

    raw = dataframe.copy()

    invalid_timestamp_rows = int(
        raw["datetime_utc"].isna().sum()
    )

    clean = raw.dropna(
        subset=["datetime_utc"]
    ).copy()

    clean["hour_utc"] = (
        clean["datetime_utc"].dt.floor("h")
    )

    duplicate_hours = int(
        clean.duplicated(
            subset=["hour_utc"],
            keep=False,
        ).sum()
    )

    hourly = (
        clean.sort_values("datetime_utc")
        .drop_duplicates(
            subset=["hour_utc"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    available_weather_columns = [
        variable
        for variable in WEATHER_VARIABLES
        if variable in hourly.columns
    ]

    missing_variables = [
        variable
        for variable in WEATHER_VARIABLES
        if variable not in hourly.columns
    ]

    missing_by_variable = {
        variable: (
            int(hourly[variable].isna().sum())
            if variable in hourly.columns
            else len(hourly)
        )
        for variable in WEATHER_VARIABLES
    }

    if missing_variables:
        complete_rows = 0
    else:
        complete_rows = int(
            hourly[available_weather_columns]
            .notna()
            .all(axis=1)
            .sum()
        )

    unique_forecast_hours = len(hourly)

    first_timestamp = (
        hourly["hour_utc"].min()
        if not hourly.empty
        else None
    )

    last_timestamp = (
        hourly["hour_utc"].max()
        if not hourly.empty
        else None
    )

    timestamps_are_hourly = False
    longest_missing_gap_hours = EXPECTED_FORECAST_HOURS

    if first_timestamp is not None:
        expected_index = pd.date_range(
            start=first_timestamp,
            periods=EXPECTED_FORECAST_HOURS,
            freq="h",
        )

        observed_index = pd.DatetimeIndex(
            hourly["hour_utc"]
        )

        missing_mask = ~expected_index.isin(
            observed_index
        )

        longest_missing_gap_hours = (
            longest_missing_run(
                pd.Series(missing_mask)
            )
        )

        timestamp_differences = (
            hourly["hour_utc"]
            .sort_values()
            .diff()
            .dropna()
        )

        timestamps_are_hourly = bool(
            timestamp_differences.eq(
                pd.Timedelta(hours=1)
            ).all()
        )

    covers_72_hours = (
        unique_forecast_hours >= EXPECTED_FORECAST_HOURS
        and longest_missing_gap_hours == 0
    )

    all_required_values_complete = (
        not missing_variables
        and complete_rows >= EXPECTED_FORECAST_HOURS
    )

    forecast_is_valid = (
        covers_72_hours
        and timestamps_are_hourly
        and all_required_values_complete
        and duplicate_hours == 0
        and invalid_timestamp_rows == 0
    )

    return {
        **base_report,
        "returned_rows": len(raw),
        "unique_forecast_hours": unique_forecast_hours,
        "complete_hours": complete_rows,
        "duplicate_hours": duplicate_hours,
        "invalid_timestamp_rows": invalid_timestamp_rows,
        "missing_variables": missing_variables,
        "missing_values_by_variable": missing_by_variable,
        "first_forecast_timestamp": (
            first_timestamp.isoformat()
            if first_timestamp is not None
            else None
        ),
        "last_forecast_timestamp": (
            last_timestamp.isoformat()
            if last_timestamp is not None
            else None
        ),
        "longest_missing_gap_hours": (
            longest_missing_gap_hours
        ),
        "covers_72_hours": covers_72_hours,
        "timestamps_are_hourly": timestamps_are_hourly,
        "all_required_values_complete": (
            all_required_values_complete
        ),
        "forecast_is_valid": forecast_is_valid,
    }

In [16]:
forecast_report = evaluate_forecast(
    dataframe=forecast_df,
)

print(
    json.dumps(
        forecast_report,
        indent=2,
    )
)

forecast_report_file = (
    OUTPUT_DIR / "forecast_weather_report.json"
)

with forecast_report_file.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        forecast_report,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"\nForecast validation report saved to: "
    f"{forecast_report_file}"
)

{
  "required_forecast_hours": 72,
  "returned_rows": 72,
  "unique_forecast_hours": 72,
  "complete_hours": 72,
  "duplicate_hours": 0,
  "invalid_timestamp_rows": 0,
  "missing_variables": [],
  "missing_values_by_variable": {
    "temperature_2m": 0,
    "relative_humidity_2m": 0,
    "dew_point_2m": 0,
    "surface_pressure": 0,
    "precipitation": 0,
    "rain": 0,
    "cloud_cover": 0,
    "visibility": 0,
    "wind_speed_10m": 0,
    "wind_direction_10m": 0,
    "wind_gusts_10m": 0
  },
  "first_forecast_timestamp": "2026-07-25T11:00:00+00:00",
  "last_forecast_timestamp": "2026-07-28T10:00:00+00:00",
  "longest_missing_gap_hours": 0,
  "covers_72_hours": true,
  "timestamps_are_hourly": true,
  "all_required_values_complete": true,
  "forecast_is_valid": true
}

Forecast validation report saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/open_meteo_validation/forecast_weather_report.json


In [17]:
validation_rules = {
    "temperature_2m": (-10, 60),
    "relative_humidity_2m": (0, 100),
    "dew_point_2m": (-20, 45),
    "surface_pressure": (850, 1100),
    "precipitation": (0, 500),
    "rain": (0, 500),
    "cloud_cover": (0, 100),
    "visibility": (0, 100000),
    "wind_speed_10m": (0, 250),
    "wind_direction_10m": (0, 360),
    "wind_gusts_10m": (0, 300),
}

range_results = []

for column, (minimum, maximum) in validation_rules.items():
    if column not in historical_weather_df.columns:
        range_results.append({
            "variable": column,
            "available": False,
            "below_minimum": None,
            "above_maximum": None,
        })
        continue

    values = historical_weather_df[column]

    range_results.append({
        "variable": column,
        "available": True,
        "missing_values": int(values.isna().sum()),
        "observed_minimum": float(values.min()),
        "observed_maximum": float(values.max()),
        "below_expected_minimum": int(
            (values < minimum).sum()
        ),
        "above_expected_maximum": int(
            (values > maximum).sum()
        ),
    })

range_report = pd.DataFrame(range_results)

display(range_report)

range_report.to_csv(
    OUTPUT_DIR / "weather_range_validation.csv",
    index=False,
)

,variable,available,missing_values,observed_minimum,observed_maximum,below_expected_minimum,above_expected_maximum
0,temperature_2m,True,0,9.2,42.1,0,0
1,relative_humidity_2m,True,0,6.0,100.0,0,0
2,dew_point_2m,True,0,-7.4,27.7,0,0
3,surface_pressure,True,0,993.6,1024.6,0,0
4,precipitation,True,0,0.0,20.0,0,0
5,rain,True,0,0.0,20.0,0,0
6,cloud_cover,True,0,0.0,100.0,0,0
7,visibility,True,9144,NaN,NaN,0,0
8,wind_speed_10m,True,0,0.0,24.3,0,0
9,wind_direction_10m,True,0,1.0,360.0,0,0


## Final Open-Meteo source decision

Historical and forecast weather will use:

- Latitude: 24.814741
- Longitude: 67.067062
- Timezone: UTC

Selected hourly variables:

- temperature_2m
- relative_humidity_2m
- dew_point_2m
- surface_pressure
- precipitation
- rain
- cloud_cover
- wind_speed_10m
- wind_direction_10m
- wind_gusts_10m

Visibility was excluded because it was missing for all historical rows.